# 07 — SHAP Interpretability

**Question:** which features are actually driving the XGBoost risk predictions? Recruiters landing on the demo see a per-prediction SHAP waterfall — this notebook proves the numbers behind it.

## Setup

In [ ]:
from pathlib import Path
import joblib, pandas as pd, numpy as np, shap, matplotlib.pyplot as plt
ART = Path('../ml/artifacts')
FIG = Path('../docs/figures'); FIG.mkdir(exist_ok=True, parents=True)
artifact = joblib.load(ART/'risk_xgb.pkl'); model = artifact['model']; feats = artifact['features']
X = pd.read_parquet('../data/processed/risk_features.parquet')[feats]

## Global explanation

In [ ]:
explainer = shap.TreeExplainer(model)
shap_vals = explainer(X.sample(500, random_state=42))
shap.summary_plot(shap_vals, show=False)
plt.tight_layout(); plt.savefig(FIG/'07_shap_summary.png', dpi=140)

## Local waterfall for one high-risk row

In [ ]:
preds = model.predict(X)
idx = int(np.argmax(preds))
shap.plots.waterfall(shap_vals[0], show=False)
plt.tight_layout(); plt.savefig(FIG/'07_shap_waterfall_example.png', dpi=140)

## Save top-contributors helper (used by the API)

In [ ]:
import json
(ART/'risk_xgb_feature_order.json').write_text(json.dumps(feats, indent=2))